In [11]:
import json
import os
import requests
import csv
import rdflib
from rdflib import Graph, URIRef, Literal, Namespace, BNode, Dataset
from rdflib.namespace import SKOS, DCTERMS, DCMITYPE, RDF, RDFS, XSD, PROV, SDO, TIME, split_uri
from SPARQLWrapper import SPARQLWrapper, CSV

from openai import OpenAI
import re

from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import networkx as nx
import pandas as pd

In [4]:
# Opening config file, the config structure is:
# {"openai_api_key":"......"}

config = open('config', 'r')
config = json.load(config)

os.environ['OPENAI_API_KEY'] = config['openai_api_key']
os.environ['GEMINI_API_KEY'] = config['gemini_api_key']
os.environ['XAI_API_KEY'] = config['xai_api_key']
os.environ['NVIDIA_API_KEY'] = config['nvidia_api_key']
os.environ['DEEPSEEK_API_KEY'] = config['deepseek_api_key']
os.environ['ANTHROPIC_API_KEY'] = config['claude_api_key']
os.environ['DASHSCOPE_API_KEY'] = config['dashscope_api_key']

In [5]:
# Load the JSON dataset
with open('benchmarks/CQs_SPARQL_ea.json', 'r') as f:
    qa = json.load(f)

# Access the dataset
qa_pairs = qa['qa_pairs']

In [6]:
def load_graph(data):
    g = Graph()
    g.parse(data=data, format="turtle")
    return g

In [7]:
def print_rdf(rdf):
    g = Graph()
    g.parse(data=rdf, format="turtle")

    for s, p, o in g:
        print(s, p, o)

In [8]:
# Namespaces
she = Namespace("https://soilwise-he.github.io/soil-health#")
agrovoc = Namespace("http://aims.fao.org/aos/agrovoc/")
agrontology = Namespace("http://aims.fao.org/aos/agrontology#")
sio = Namespace("http://semanticscience.org/resource/")
glosis_lh = Namespace("http://w3id.org/glosis/model/layerhorizon/")
glosis_sp = Namespace("http://w3id.org/glosis/model/siteplot/")
qudt = Namespace("http://qudt.org/schema/qudt/")
unit = Namespace("http://qudt.org/vocab/unit/")
iso11074 = Namespace("https://data.geoscience.earth/ncl/ISO11074v2025/")
obo = Namespace("http://purl.obolibrary.org/obo/")
wdt = Namespace("http://www.wikidata.org/prop/direct/")
biolink = Namespace("https://w3id.org/biolink/vocab/")
afox = Namespace("http://purl.allotrope.org/ontologies/property#")
afor = Namespace("http://purl.allotrope.org/ontologies/result#")
sorelsc = Namespace("http://sweetontology.net/relaSci/")
sorelpr = Namespace("http://sweetontology.net/relaProvenance/")
sohuj = Namespace("http://sweetontology.net/humanJurisdiction/")
sorelph = Namespace("http://sweetontology.net/relaPhysical/")
sorelm = Namespace("http://sweetontology.net/relaMath/")
sorepsg = Namespace("http://sweetontology.net/reprSpaceGeometry/")
bao = Namespace("http://www.bioassayontology.org/bao#")
repr = Namespace("https://w3id.org/reproduceme#")
sorelch = Namespace("http://sweetontology.net/relaChemical/")
sorelsp = Namespace("http://sweetontology.net/relaSpace/")
om = Namespace("http://www.ontology-of-units-of-measure.org/resource/om-2/")
gemet = Namespace("http://www.eionet.europa.eu/gemet/concept/")
inrae = Namespace("http://opendata.inrae.fr/thesaurusINRAE/")

### QA over soil health KG by RAG

#### Entity extraction from query

In [7]:
system_prompt_ee = """You are a highly specialized entity extraction agent. Your primary function is to analyze a user's natural language query and extract all mentions of entities (people, organizations, dates, products, technical concepts, geographical locations, etc.). The ultimate goal is to use these entities to sample a subgraph from a domain knowledge graph.

Your task is to identify entities and provide a standardized canonical term for each entity that would exist in a formal knowledge graph.

**# Extraction & Standardization Rules**

1. **Identify Entities:** Carefully read the user's query and identify all possible entities.
2. **Preserve Original Text:** For each entity found, you must capture the exact text as it appeared in the query.
3. **Standardize Term:** For each entity, you must also provide a standardized, canonical term. This term should be the most common or formal name for the entity, as one might find in a knowledge base or encyclopedia.

**# Output Format**

Your output MUST be a valid JSON array of objects."""

In [8]:
prompt_cq = f"""Now please extract all entities from the following query:
'{qa_pairs[3]['competency_question']}'"""

In [10]:
client = OpenAI()

completion = client.chat.completions.create(
    model="gpt-4.1",
    messages=[
        {"role": "system", "content": system_prompt_ee},
        {"role": "user", "content": prompt_cq}
    ],
    response_format={ 
        "type": "json_schema", 
        "json_schema": {
            "name": "entity_extraction",
            "schema": {
                "type": "object",
                "properties": {
                    "user_query": {
                        "type": "string",
                        "description": "The natural language query provided by the user for which entities need to be extracted.",
                        "minLength": 1
                        },
                    "extracted_entities": {
                        "type": "array",
                        "description": "A list of entities extracted from the user query.",
                        "items": {
                            "type": "object",
                            "properties": {
                                "entity": {
                                    "type": "string",
                                    "description": "The original entity extracted from the query."
                                    },
                                "standardized_term": {
                                    "type": "string",
                                    "description": "A standardized term corresponding to the original entity, possibly inferred."
                                    }
                                  },
                          "required": [
                              "entity",
                              "standardized_term"
                              ],
                          "additionalProperties": False
                        }
                      }
                    },
                "required": [
                    "user_query",
                    "extracted_entities"
                    ],
                "additionalProperties": False
              },
            "strict": True
        }
    }
)

print(completion.choices[0].message.content)

{"user_query":"What is the difference between soil health and soil quality?","extracted_entities":[{"entity":"soil health","standardized_term":"Soil Health"},{"entity":"soil quality","standardized_term":"Soil Quality"}]}


In [11]:
print(json.loads(completion.choices[0].message.content)["extracted_entities"])

[{'entity': 'soil health', 'standardized_term': 'Soil health'}, {'entity': 'soil quality', 'standardized_term': 'Soil quality'}]


#### Sample subgraph from soil health KG

In [12]:
def sample_subgraph_from_json(json_string: str, turtle_file_path: str) -> str:
    """
    Samples a subgraph from an RDF graph based on terms in a JSON object,
    with special handling for blank nodes and ignoring mapping predicates.

    Args:
        json_string: A string containing the JSON data with entities and terms.
        turtle_file_path: The file path to the RDF knowledge graph in Turtle format.

    Returns:
        A string representing the sampled subgraph in Turtle format.
    """
    # --- Step 1: Get all terms from JSON and convert to lowercase ---
    try:
        # The replace call handles JSON strings that use single quotes
        data = json.loads(json_string.replace("'", "\""))["extracted_entities"]
        search_terms = set()
        for item in data:
            if 'entity' in item:
                search_terms.add(item['entity'].lower())
            if 'standardized_term' in item:
                search_terms.add(item['standardized_term'].lower())
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON: {e}")
        return ""
        
    print(f"Searching for terms: {search_terms}\n")

    # --- Step 2: Load the knowledge graph from file and find matched nodes ---
    g = Graph()
    try:
        if not os.path.exists(turtle_file_path):
            return f"# Error: File not found at '{turtle_file_path}'"
        g.parse(source=turtle_file_path, format="turtle")
    except Exception as e:
        print(f"Error parsing Turtle file: {e}")
        return ""

    matched_nodes = set()
    label_properties = [SKOS.prefLabel, SKOS.altLabel]

    for s, p, o in g:
        if p in label_properties and isinstance(o, Literal) and o.value.lower() in search_terms:
            print(f"Found match: Subject <{s}> has label '{o}'")
            matched_nodes.add(s)

    if not matched_nodes:
        print("No matching nodes found in the knowledge graph.")
        return "# No matching nodes found."

    print(f"\nMatched nodes: {[str(node) for node in matched_nodes]}\n")

    # --- Step 3: Sample the subgraph with special blank node and predicate handling ---
    subgraph = Graph()
    for prefix, namespace in g.namespace_manager.namespaces():
        subgraph.bind(prefix, namespace)

    # Define the set of predicates to ignore
    ignored_predicates = {SKOS.exactMatch, SKOS.closeMatch}
    b_nodes_to_expand = set()

    # First pass: handle direct connections to matched nodes
    for s, p, o in g:
        # Case 1: The subject is a matched node.
        if s in matched_nodes:
            # Add the triple ONLY if the predicate is not ignored
            if p not in ignored_predicates:
                subgraph.add((s, p, o))
                # If the object is a blank node, mark it for expansion.
                if isinstance(o, BNode):
                    b_nodes_to_expand.add(o)

        # Case 2: The object is a matched node.
        elif isinstance(o, URIRef) and o in matched_nodes:
            # Per rule: Ignore triples where a blank node links TO a matched node.
            # Also ignore triples with specified mapping predicates.
            if not isinstance(s, BNode) and p not in ignored_predicates:
                subgraph.add((s, p, o))

    # Second pass: expand the collected blank nodes one level deeper.
    if b_nodes_to_expand:
        print(f"Expanding blank nodes: {[str(bnode) for bnode in b_nodes_to_expand]}\n")
        for s, p, o in g:
            if s in b_nodes_to_expand:
                # Also apply predicate filter here for consistency
                if p not in ignored_predicates:
                    subgraph.add((s, p, o))
            
    # --- Step 4: Return the subgraph in Turtle format ---
    return subgraph.serialize(format="turtle")


In [13]:
# Write the sample KG to a local file for the example
ttl_file_name = "soil_health_KG.ttl"

# Run the sampling function using the file path
sampled_turtle = sample_subgraph_from_json(completion.choices[0].message.content, ttl_file_name)

# Print the final result
print("--- Sampled Subgraph (Turtle Format) ---")
print(sampled_turtle)

Searching for terms: {'soil health', 'soil quality'}

Found match: Subject <https://soilwise-he.github.io/soil-health#SoilHealth> has label 'soil health'
Found match: Subject <https://soilwise-he.github.io/soil-health#SoilQuality> has label 'soil quality'

Matched nodes: ['https://soilwise-he.github.io/soil-health#SoilHealth', 'https://soilwise-he.github.io/soil-health#SoilQuality']

--- Sampled Subgraph (Turtle Format) ---
@prefix af-x: <http://purl.allotrope.org/ontologies/property#> .
@prefix agrontology: <http://aims.fao.org/aos/agrontology#> .
@prefix bao: <http://www.bioassayontology.org/bao#> .
@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix obo: <http://purl.obolibrary.org/obo/> .
@prefix schema: <https://schema.org/> .
@prefix she: <https://soilwise-he.github.io/soil-health#> .
@prefix skos: <http://www.w3.org/2004/02/skos/core#> .

she:GAEC schema:inSupportOf she:SoilQuality .

she:RuralDevelopmentMeasures schema:inSupportOf she:SoilQuality .

she:SoilProtectionProcedu

In [14]:
# --- Configuration ---
# Define the new namespace for your human-readable terms
SHE = Namespace("https://soilwise-he.github.io/soil-health#")

# File paths for mapping files
CLASSES_CSV_PATH = 'ontologies/classes.csv'
PROPERTIES_CSV_PATH = 'ontologies/properties.csv'

# --- Helper Functions for Case Conversion ---

def to_pascal_case(text: str) -> str:
    """Converts a string to PascalCase.
    Example: "soil health" -> "SoilHealth"
    """
    if not isinstance(text, str):
        return ""
    return ''.join(word.capitalize() for word in re.split(r'[\s_-]+', text))

def to_camel_case(text: str) -> str:
    """Converts a string to camelCase.
    Example: "has measurement" -> "hasMeasurement"
    """
    if not isinstance(text, str):
        return ""
    parts = re.split(r'[\s_-]+', text)
    if not parts:
        return ""
    return parts[0].lower() + ''.join(word.capitalize() for word in parts[1:])

# --- Main Transformation Logic ---

def transform_rdf_graph(input_turtle_string: str, class_map: dict, prop_map: dict) -> str:
    """
    Loads an RDF graph from a string, replaces URIs based on the provided maps,
    and returns the transformed graph as a Turtle string.
    """
    # 1. Initialize graphs
    original_graph = Graph()
    transformed_graph = Graph()

    print("Parsing input Turtle string...")
    try:
        # Use `data` parameter to parse from a string
        original_graph.parse(data=input_turtle_string, format="turtle")
        print(f"Successfully parsed. Found {len(original_graph)} triples.")
    except Exception as e:
        print(f"Error parsing the Turtle string: {e}")
        return "" # Return an empty string on error

    # Bind the new 'she' prefix to the transformed graph for cleaner output
    transformed_graph.bind("she", SHE)
    # It's good practice to also bind other common prefixes from the original graph
    for prefix, namespace in original_graph.namespace_manager.namespaces():
        transformed_graph.bind(prefix, namespace)

    # 2. Iterate through all triples and transform them
    print("Transforming URIs...")
    for s, p, o in original_graph:
        new_s = s
        new_p = p
        new_o = o

        # Transform subject (if it's a URI we have a mapping for)
        if isinstance(s, URIRef) and str(s) in class_map:
            new_s = class_map[str(s)]

        # Transform predicate (if it's a property URI, and not rdf:type)
        if isinstance(p, URIRef) and p != RDF.type and str(p) in prop_map:
            new_p = prop_map[str(p)]

        # Transform object (if it's a URI we have a mapping for)
        if isinstance(o, URIRef):
            if str(o) in class_map:
                new_o = class_map[str(o)]
            elif str(o) in prop_map:
                # This case is less common but possible
                new_o = prop_map[str(o)]

        # Add the (potentially modified) triple to the new graph
        transformed_graph.add((new_s, new_p, new_o))

    # 3. Serialize the new graph to a string and return it
    print("Serialization complete!")
    return transformed_graph.serialize(format="turtle")


def main():
    """
    Main function to run the script.
    """
    # Check if required CSV files exist
    if not all(os.path.exists(f) for f in [CLASSES_CSV_PATH, PROPERTIES_CSV_PATH]):
        print("Error: One or more required files are missing.")
        print(f"Please ensure '{CLASSES_CSV_PATH}' and '{PROPERTIES_CSV_PATH}' are in the same directory as this script.")
        return

    # 1. Load mappings from CSV files
    print("Loading URI mappings from CSV files...")
    try:
        # Assuming CSVs have no header and URI is 2nd col (idx 1), label is 3rd col (idx 2)
        classes_df = pd.read_csv(CLASSES_CSV_PATH, header=None, usecols=[1, 2])
        properties_df = pd.read_csv(PROPERTIES_CSV_PATH, header=None, usecols=[1, 2])

        # Create the mapping dictionaries
        class_mappings = {row[1]: SHE[to_pascal_case(row[2])] for _, row in classes_df.iterrows()}
        property_mappings = {row[1]: SHE[to_camel_case(row[2])] for _, row in properties_df.iterrows()}

        print(f"Loaded {len(class_mappings)} class mappings and {len(property_mappings)} property mappings.")

    except FileNotFoundError as e:
        print(f"Error: Could not find the CSV file - {e.filename}")
        return
    except Exception as e:
        print(f"An error occurred while reading the CSV files: {e}")
        return

    # 2. Run the transformation and get the result as a string
    transformed_turtle_string = transform_rdf_graph(sampled_turtle, class_mappings, property_mappings)

    # 3. Print the transformed string
    print("\n--- Transformed Turtle Data ---")
    print(transformed_turtle_string)

    return transformed_turtle_string

if __name__ == "__main__":
    sampled_subgraph = main()

Loading URI mappings from CSV files...
Loaded 20 class mappings and 206 property mappings.
Parsing input Turtle string...
Successfully parsed. Found 78 triples.
Transforming URIs...
Serialization complete!

--- Transformed Turtle Data ---
@prefix she: <https://soilwise-he.github.io/soil-health#> .

she:GAEC she:inSupportOf she:SoilQuality .

she:RuralDevelopmentMeasures she:inSupportOf she:SoilQuality .

she:SoilProtectionProcedures she:focusesOn she:SoilHealth .

she:SoilProtectionRegulations she:focusesOn she:SoilHealth .

she:EcologicalIndicators she:hasNarrower she:SoilHealth .

she:InfrastructurePlatform she:isUseOf she:SoilQuality .

she:SOCContents she:isIndicatorOf she:SoilHealth .

she:SoilBiologicalProperties she:hasRelated she:SoilQuality .

she:SoilCapability she:hasRelated she:SoilQuality .

she:SoilChemicalProperties she:hasRelated she:SoilQuality .

she:SoilDegradation she:hasRelated she:SoilQuality .

she:SoilErosion she:negativelyRegulates she:SoilQuality .

she:SoilFu

#### RAG based on sampled graph

In [15]:
system_prompt_rag = """### Knowledge Graph Question Answering

**Your Role:** You are a highly specialized Question Answering AI. Your sole purpose is to answer a user's query based *only* on the information contained within a given knowledge graph segment. You must not use any external knowledge or information you were trained on.

**Instructions:**

1. **Analyze the User Query:** First, carefully examine the user's question to identify the key entities and the specific relationship or piece of information being asked for.
2. **Consult the Knowledge Graph:** You will be provided with a knowledge graph segment serialized in **Turtle (TTL) format**. This is your **only** source of truth.
3. **Synthesize the Answer:**
   * If you can find the information by traversing the relationships in the provided graph, formulate a concise, natural-language answer based directly on those facts.
   * Cite the specific triple(s) you used to derive the answer, referencing the subject, predicate, and object.
4. **Handle Missing Information:**
   * If the knowledge graph segment does not contain the entities or relationships needed to answer the query, you **must** respond with the following exact phrase: `The provided knowledge graph does not cover this information, therefore I cannot answer.`
   * Do not apologize, attempt to search elsewhere, or use your general knowledge.

**Example 1: Information Found**

**Input:**

* **User Query:** "Who directed the movie Inception?"
* **Knowledge Graph Segment:**
  ```turtle
  @prefix ex: <http://example.org/entity/> .
  @prefix prop: <http://example.org/property/> .
  
  ex:Inception prop:has_director ex:Christopher_Nolan ;
               prop:has_genre "Science Fiction" .
  
  ex:Christopher_Nolan prop:born_in "London" .
  ```

**Correct Output:**

Christopher Nolan directed the movie Inception.

Source: ex:Inception prop:has\_director ex:Christopher\_Nolan

**Example 2: Information Not Found**

**Input:**

* **User Query:** "What is the budget of the movie Inception?"
* **Knowledge Graph Segment:**
  ```turtle
  @prefix ex: <http://example.org/entity/> .
  @prefix prop: <http://example.org/property/> .
  
  ex:Inception prop:has_director ex:Christopher_Nolan ;
               prop:has_genre "Science Fiction" .
  
  ex:Christopher_Nolan prop:born_in "London" .
  ```

**Correct Output:**

The provided knowledge graph does not cover this information, therefore I cannot answer."""

In [16]:
prompt_rag = f"""Now please answer the following question:
'{qa_pairs[3]['competency_question']}',
based on the provided knowledge graph segment:
'{sampled_subgraph}'"""

In [17]:
client = OpenAI()

completion = client.chat.completions.create(
    model="gpt-4.1",
    messages=[
        {"role": "system", "content": system_prompt_rag},
        {"role": "user", "content": prompt_rag}
    ],
    response_format={
        "type": "json_schema", 
        "json_schema": {
            "name": "kg_qa_answer",
            "schema": {
                "type": "object",
                "properties": {
                    "answer": {
                        "type": "string",
                        "description": "Concise natural-language answer synthesized strictly from facts in the given knowledge graph segment. Respond `The provided knowledge graph does not cover this information, therefore I cannot answer.` when the knowledge graph segment does not contain relevant triples."
                    },
                    "evidence_triples": {
                        "type": "array",
                        "description": "List of RDF triples (as strings, in Turtle syntax) that directly support or justify the answer. Leave this empty when the answer is `The provided knowledge graph does not cover this information, therefore I cannot answer.`",
                        "items": {
                            "type": "string",
                            "minLength": 0,
                            "description": "A single triple in valid Turtle (TTL) format."
                        }
                    }
                },
                "required": [
                    "answer",
                    "evidence_triples"
                ],
                "additionalProperties": False
            },
            "strict": True
        }
    }
)

print(completion.choices[0].message.content)

{"answer":"Soil health is defined as the actual capacity of a soil to perform its core functions and to provide ecosystem services, while soil quality is defined as the inherent capacity which provides the basic frame within which a soil evolves. In summary, soil health emphasizes the current functional state and ecosystem services of the soil, whereas soil quality focuses on the soil's inherent properties that determine how it develops over time.","evidence_triples":["she:SoilHealth a she:Concept ;\n    she:definition \"Soil health is defined as the actual capacity of a soil to perform its core functions and to provide ecosystem services.\"@en,\n        \"The definition of soil quality and soil health is the capacity of a soil to function within ecosystems and land-use boundaries to sustain biological productivity, maintain environmental quality, and promote plant and animal health.\"@en ;\n    she:hasBroader she:EcologicalIndicators ;\n    she:hasIndicator she:SOCContents ;\n    she:

In [18]:
print(json.loads(completion.choices[0].message.content)["answer"])

Soil health is defined as the actual capacity of a soil to perform its core functions and to provide ecosystem services, while soil quality is defined as the inherent capacity which provides the basic frame within which a soil evolves. In summary, soil health emphasizes the current functional state and ecosystem services of the soil, whereas soil quality focuses on the soil's inherent properties that determine how it develops over time.


In [21]:
print(f"Reference: {qa_pairs[3]['expected_answer']} Model answer: {json.loads(completion.choices[0].message.content)['answer']}")

Reference: Soil health is defined as the actual capacity of a soil to perform its core functions and to provide ecosystem services, and soil quality as the inherent capacity which provides the basic frame within which a soil evolves. Model answer: Soil health is defined as the actual capacity of a soil to perform its core functions and to provide ecosystem services, while soil quality is defined as the inherent capacity which provides the basic frame within which a soil evolves. In summary, soil health emphasizes the current functional state and ecosystem services of the soil, whereas soil quality focuses on the soil's inherent properties that determine how it develops over time.


#### Evaluation using LLMs

In [22]:
def evaluate_qa_with_llm_judge(reference_answer, model_answer, api_key=None, model="gpt-4o-2024-08-06"):
    """
    Evaluate a QA pair using LLM-as-a-judge with OpenAI's grader API.
    
    Args:
        reference_answer (str): The ground truth answer
        model_answer (str): The model's generated answer
        api_key (str, optional): OpenAI API key (uses env var if None)
        model (str): Model to use for evaluation
    
    Returns:
        dict: Evaluation results including score, metadata, and pass/fail status
    """
    
    if api_key is None:
        api_key = os.environ.get("OPENAI_API_KEY")
    
    if not api_key:
        raise ValueError("OpenAI API key not found. Set OPENAI_API_KEY environment variable or pass api_key parameter.")
    
    headers = {"Authorization": f"Bearer {api_key}"}
    
    # Define the grader
    grader = {
       "type": "score_model",
       "name": "soil_health_qa_grader",
       "input": [
            {
                "role": "system",
                "content": """You are an expert grader evaluating question-answering performance in the soil health domain. 
                
                Scoring criteria:
                - Score 1.0: Reference and model answer are semantically equivalent, covering the same key concepts and facts
                - Score 0.8-0.9: Model answer covers most key concepts but may miss some details or have minor inaccuracies  
                - Score 0.6-0.7: Model answer covers core concepts but has some gaps or inaccuracies
                - Score 0.4-0.5: Model answer covers some key concepts but has significant gaps or inaccuracies
                - Score 0.2-0.3: Model answer has major errors or covers very few key concepts
                - Score 0.0-0.1: Model answer is completely wrong, irrelevant, or states it cannot answer
                
                Focus on semantic meaning and factual accuracy rather than exact wording.
                Consider the completeness, accuracy, and relevance of the information provided."""
            },
            {
                "role": "user",
                "content": f"Reference Answer: {reference_answer}\n\nModel Answer: {model_answer}\n\nPlease evaluate how well the model answer matches the reference answer and provide a score between 0 and 1."
            }
       ],
       "pass_threshold": 0.5,
       "model": model,
       "range": [0, 1]
    }
    
    # Validate the grader
    payload = {"grader": grader}
    response = requests.post(
        "https://api.openai.com/v1/fine_tuning/alpha/graders/validate",
        json=payload,
        headers=headers
    )
    
    if response.status_code != 200:
        return {
            "error": "Grader validation failed",
            "status_code": response.status_code,
            "response": response.text
        }
    
    # Run the evaluation
    payload = {
        "grader": grader,
        "item": {
           "reference_answer": reference_answer
        },
        "model_sample": model_answer
    }
    
    response = requests.post(
        "https://api.openai.com/v1/fine_tuning/alpha/graders/run",
        json=payload,
        headers=headers
    )
    
    if response.status_code != 200:
        return {
            "error": "Evaluation failed",
            "status_code": response.status_code,
            "response": response.text
        }
    
    result = response.json()
    score = result.get("reward", 0.0)
    metadata = result.get("metadata", {})
    
    return {
        "score": score,
        "max_score": 1.0,
        "pass_threshold": 0.5,
        "pass": score >= 0.5,
        "execution_time": metadata.get("execution_time", 0),
        "token_usage": metadata.get("token_usage", {}),
        "reference_answer": reference_answer,
        "model_answer": model_answer,
        "errors": metadata.get("errors", {}),
        "raw_result": result
    }

# Example usage with your current QA pair
print("=== USING THE EVALUATION FUNCTION ===")
result = evaluate_qa_with_llm_judge(
    reference_answer=qa_pairs[3]['expected_answer'],
    model_answer=json.loads(completion.choices[0].message.content)['answer']
)

if "error" in result:
    print(f"Error: {result['error']}")
    print(f"Details: {result.get('response', 'N/A')}")
else:
    print(f"Score: {result['score']:.3f}/{result['max_score']}")
    print(f"Status: {'PASS' if result['pass'] else 'FAIL'} (threshold: {result['pass_threshold']})")
    print(f"Execution time: {result['execution_time']:.2f}s")
    print(f"Total tokens used: {result['token_usage'].get('total_tokens', 'N/A')}")
    
    # Check for any errors during evaluation
    errors = result['errors']
    if any(errors.values()):
        print("\nWarnings/Errors detected:")
        for error_type, occurred in errors.items():
            if occurred:
                print(f"  - {error_type}")
    else:
        print("\nNo evaluation errors detected.")

=== USING THE EVALUATION FUNCTION ===
Score: 1.000/1.0
Status: PASS (threshold: 0.5)
Execution time: 2.98s
Total tokens used: 687

No evaluation errors detected.


### QA over soil health KG by SPARQL

In [2]:
with open('benchmarks/CQs_SPARQL_ea.json', 'r') as f:
    qa = json.load(f)
query = qa['qa_pairs'][23]['sparql_query']

print(query)

PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX agrontology: <http://aims.fao.org/aos/agrontology#>
PREFIX qudt: <http://qudt.org/schema/qudt/>

SELECT ?item ?property ?value
WHERE {
  {
    ?somConcept skos:prefLabel "soil organic matter"@en .
    ?somConcept skos:definition ?somDefinition .
    BIND("Soil Organic Matter" AS ?item)
    BIND(skos:definition AS ?property)
    BIND(?somDefinition AS ?value)
  }
  UNION
  {
    ?somConcept skos:prefLabel "soil organic matter"@en .
    ?somConcept agrontology:isMadeFrom ?component .
    ?component skos:prefLabel ?componentLabel .
    BIND("Soil Organic Matter" AS ?item)
    BIND(agrontology:isMadeFrom AS ?property)
    BIND(?componentLabel AS ?value)
  }
  UNION
  {
    ?conversionConcept skos:prefLabel "SOC to SOM conversion"@en .
    ?conversionConcept qudt:conversionMultiplier ?factorNode .

    ?factorNode qudt:numericValue ?factorValue .
    ?factorNode skos:definition ?factorDefinition .

    BIND(CONCAT("Factor: ", STR(?f

In [17]:
def sparql_to_dataframe(graph, query):
    """
    Executes a SPARQL query against an rdflib Graph and returns the results
    as a pandas DataFrame.
    """
    # Execute the query
    results = graph.query(query)

    # Extract variable names from the query results
    variables = [str(var) for var in results.vars]

    # Convert the results to a list of dictionaries
    data = []
    for row in results:
        row_dict = {}
        for var in variables:
            # Get the value for each variable in the current row
            value = getattr(row, var) if hasattr(row, var) else None
            # Convert the value to a string, handling None values
            row_dict[var] = str(value) if value is not None else "" # Use empty string for None to match CSV output
        data.append(row_dict)

    # Create and return the DataFrame
    return pd.DataFrame(data)

# Initialize an empty graph
g = Graph()
g.parse("soil_health_KG.ttl", format="turtle")

df = sparql_to_dataframe(g, query)
print(df.to_csv(index=False, quoting=csv.QUOTE_ALL))

"item","property","value"
"Factor: 1.724","http://www.w3.org/2004/02/skos/core#definition","Historically, a factor of 1.724 has been used to convert SOC to SOM, based on the assumption that organic matter is 58% carbon."
"Factor: 2","http://www.w3.org/2004/02/skos/core#definition","A factor of 2 would often be more accurate, especially in the case of soil layers rich in organic matter, such as on forest floors, because of differences related to the different stages of decomposition and mineralisation."
"Soil Organic Matter","http://aims.fao.org/aos/agrontology#isMadeFrom","animal residues"
"Soil Organic Matter","http://aims.fao.org/aos/agrontology#isMadeFrom","attached cations"
"Soil Organic Matter","http://aims.fao.org/aos/agrontology#isMadeFrom","attached ions"
"Soil Organic Matter","http://aims.fao.org/aos/agrontology#isMadeFrom","basic elements"
"Soil Organic Matter","http://aims.fao.org/aos/agrontology#isMadeFrom","microbial biomass"
"Soil Organic Matter","http://aims.fao.org/aos/

In [18]:
sparql = SPARQLWrapper("https://repository.soilwise-he.eu/sparql/")
sparql.setQuery(query)
sparql.setReturnFormat(CSV)
results = sparql.queryAndConvert()

print(results.decode('utf-8'))

"item","property","value"
"Factor: 1.723999977111816","http://www.w3.org/2004/02/skos/core#definition","Historically, a factor of 1.724 has been used to convert SOC to SOM, based on the assumption that organic matter is 58% carbon."
"Factor: 2","http://www.w3.org/2004/02/skos/core#definition","A factor of 2 would often be more accurate, especially in the case of soil layers rich in organic matter, such as on forest floors, because of differences related to the different stages of decomposition and mineralisation."
"Soil Organic Matter","http://aims.fao.org/aos/agrontology#isMadeFrom","animal residues"
"Soil Organic Matter","http://aims.fao.org/aos/agrontology#isMadeFrom","attached cations"
"Soil Organic Matter","http://aims.fao.org/aos/agrontology#isMadeFrom","attached ions"
"Soil Organic Matter","http://aims.fao.org/aos/agrontology#isMadeFrom","basic elements"
"Soil Organic Matter","http://aims.fao.org/aos/agrontology#isMadeFrom","microbial biomass"
"Soil Organic Matter","http://aims.